# Milestone 2: Skill Extraction, Similarity and Evaluation

Run cells in order with `notebooks/` as the working directory. This notebook loads the saved processed CSVs and calls functions from `src/`. `partial` supplies shared inputs to an imported function so later calls keep their original shape. Outputs have been cleared; no analysis was rerun.

## 1. Setup and Data Loading

### Import libraries and project functions

In [ ]:
# Import libraries and reusable project functions.
import ast
import sys
from functools import partial
from pathlib import Path

import pandas as pd

# Run this notebook with notebooks/ as the working directory.
sys.path.insert(0, str(Path.cwd().parent))
from src.skill_extraction import (
    build_skill_dictionary,
    create_skill_extractor,
    extract_skills,
    prepare_optimized_extractor,
    optimize_skill_dictionary,
    create_optimized_extractor,
    extract_skills_optimized,
)
from src.similarity import compare_skills, filter_generic_skills, calculate_skill_match
from src.pipeline import analyze_match
from src.evaluation import (
    evaluate_single_resume,
    evaluate_resumes,
    evaluate_optimized_resumes,
    compare_evaluation_results,
)


### Load the processed resumes and jobs

In [ ]:
cv = pd.read_csv("../data/processed/final_cleaned_cv.csv")
jobs = pd.read_csv("../data/processed/final_cleaned_jobs.csv")

print(cv.shape)
print(jobs.shape)


### Convert resume skill labels into lists

In [ ]:
# Convert resume skills from text into Python lists
cv["skills_list"] = cv["clean_skills"].apply(ast.literal_eval)

cv[["target_position", "skills_list"]].head()


### Convert job skill labels into lists

In [ ]:
# Convert job skills from text into Python lists

jobs["skills_list"] = jobs["clean_skills"].apply(ast.literal_eval)

jobs[["job_title", "skills_list"]].head()


## 2. AI/NLP Model: Skill Extraction

### Build the skill dictionary

In [ ]:
skill_dictionary = build_skill_dictionary(cv, jobs)


### Create the EntityRuler and prepare skill extraction

In [ ]:
nlp = create_skill_extractor(skill_dictionary)
extract_skills = partial(extract_skills, nlp=nlp)


### Extract skills from one resume

In [ ]:
# Test skill extraction on one resume

extracted_resume_skills = extract_skills(
    cv.loc[0, "processed_resume"]
)

print("Extracted Resume Skills:")
print(extracted_resume_skills)


### Extract skills from one job description

In [ ]:
# Extract skills from one job description

extracted_job_skills = extract_skills(
    jobs.loc[0, "processed_job"]
)

print("Extracted Job Skills:")
print(extracted_job_skills)


## 3. Matched Skills, Missing Skills and Skill Percentage

### Compare the extracted resume and job skills

In [ ]:
matched_skills, missing_skills = compare_skills(
    extracted_resume_skills, extracted_job_skills
)


### Remove generic terms from the extracted skills

In [ ]:
extracted_resume_skills, extracted_job_skills = filter_generic_skills(
    extracted_resume_skills, extracted_job_skills
)


### Compare the filtered skills

In [ ]:
matched_skills, missing_skills = compare_skills(
    extracted_resume_skills, extracted_job_skills
)


### Calculate the skill-match percentage

In [ ]:
skill_match_percentage = calculate_skill_match(matched_skills, extracted_job_skills)


## 4. TF-IDF + Cosine Similarity

### Prepare the optimized extractor for the example analysis

In [ ]:
optimized_nlp = prepare_optimized_extractor(cv, jobs)
extract_skills_optimized = partial(extract_skills_optimized, optimized_nlp=optimized_nlp)
analyze_match = partial(
    analyze_match,
    cv=cv,
    jobs=jobs,
    extract_skills_optimized=extract_skills_optimized,
)


### Analyze one resume against one job

In [ ]:
# Test the analyzer

analyze_match(3, 3)


## 5. Baseline Evaluation

### Measure precision, recall and F1 for one resume

In [ ]:
precision, recall, f1 = evaluate_single_resume(cv, extract_skills)


### Evaluate the baseline extractor across all resumes

In [ ]:
precision_scores, recall_scores, f1_scores = evaluate_resumes(cv, extract_skills)


## 6. Dictionary Optimization and Final Evaluation

### Filter the complete skill dictionary

In [ ]:
optimized_skill_dictionary = optimize_skill_dictionary(skill_dictionary)


### Rebuild the optimized EntityRuler

In [ ]:
optimized_nlp = create_optimized_extractor(optimized_skill_dictionary)
extract_skills_optimized = partial(extract_skills_optimized, optimized_nlp=optimized_nlp)
analyze_match = partial(analyze_match, extract_skills_optimized=extract_skills_optimized)


### Evaluate the optimized extractor across all resumes

In [ ]:
optimized_precision, optimized_recall, optimized_f1 = evaluate_optimized_resumes(
    cv, extract_skills_optimized
)


### Compare the original baseline and optimized metrics

In [ ]:
compare_evaluation_results(optimized_precision, optimized_recall, optimized_f1)
